# Local ART-E Email Task

This notebook is a lightweight ART-E companion task. It uses a deterministic local inbox so the search/read/answer contract can be inspected without API keys, GPUs, W&B, or remote email infrastructure.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "email_task.py").exists():
    sys.path.insert(0, str(cwd))
elif (cwd / "examples" / "art_e_local" / "email_task.py").exists():
    sys.path.insert(0, str(cwd / "examples" / "art_e_local"))

from email_task import SCENARIOS, evaluate_scripted_baseline, read_email, score_answer, search_emails

## Task Fixtures

Each scenario asks a question over a local inbox. A correct answer must contain the expected answer and cite the supporting message ID.

In [ ]:
for scenario in SCENARIOS:
    print(f"{scenario.id}: {scenario.question}")
    print(f"  expected: {scenario.expected_answer}")
    print(f"  citations: {', '.join(scenario.expected_message_ids)}")

## Search and Read Tools

A rollout can expose these functions as tools. The example keeps them deterministic so reward regressions are easy to review.

In [ ]:
scenario = SCENARIOS[1]
results = search_emails(scenario.inbox, "Northwind final", scenario.query_date)
[(email.id, email.subject) for email in results]

In [ ]:
message = read_email(results[0].id)
message

## Offline Baseline

The scripted baseline is intentionally simple. Its job is to verify the fixtures, scoring, and citation contract before an ART rollout or RULER reward is attached.

In [ ]:
for result in evaluate_scripted_baseline():
    print(result)

## Adapting to ART

To turn this into a trainable ART example, wrap `search_emails`, `read_email`, and `score_answer` in a rollout that records tool calls on an `art.Trajectory`. The local fixtures give reviewers a fast contract test before running serverless training.